 ## Updates made
 - Rotation-aware quantization: before quantizing a large 2D weight matrix, the exporter can apply a deterministic signed Hadamard block rotation.
    This spreads outlier energy across dimensions so int8 quantization wastes less dynamic range on a few columns. After dequantization, the inverse
    rotation is applied, so the model still uses weights in the original basis.
  - Better Hessian approximations: the exporter now generates self-calibration sequences, runs them back through the model, and collects per-layer
    activation second-moment matrices X^T X for large CastedLinear weights. That gives a GPTQ-style curvature proxy, so quantization error is measured
    in directions the model actually uses, instead of only plain weight-space MSE.
  - Mixed-precision placement in the stronger GPTQ sense: instead of only keeping small/control tensors in float, the exporter now estimates a Hessian-
    weighted penalty for quantizing each large tensor and spends a small explicit extra-byte budget on the most sensitive ones, keeping those in fp16.
    That is a sensitivity-driven allocator rather than a size-only passthrough rule.
  - searches a grid over:
      - SELF_CALIB_CANDIDATE_PERCENTILES
      - SEARCH_ROTATION_OPTIONS
      - SEARCH_MIXED_PRECISION_BUDGETS
      - SEARCH_MIXED_PRECISION_MAX_TENSORS
  - records exact compressed bytes for every candidate
  - builds a Pareto frontier in total_submission_bytes vs calib_loss
  - runs full roundtrip validation only on the frontier subset
  - selects the final artifact by roundtrip_val_loss under SEARCH_TARGET_TOTAL_BYTES

  Writes artifact tables to:

  - colab/2026-04-06_QuantExport3_RotationAware_GPTQMix/logs/artifact_size_table.tsv
  - colab/2026-04-06_QuantExport3_RotationAware_GPTQMix/logs/roundtrip_quality_table.tsv

  There is also:

  - colab/2026-04-06_QuantExport3_RotationAware_GPTQMix/logs/search_frontier.tsv

  The tables contain:

  - artifact_size_table.tsv: every searched candidate, with clip percentile, rotation choice, mixed-precision budget/cap, payload bytes, compressed
    bytes, total submission bytes, calibration loss, and flags for frontier_eval and selected
  - roundtrip_quality_table.tsv: only the frontier candidates that were fully evaluated, with roundtrip_val_loss, roundtrip_val_bpb, eval_time_ms,
    bytes, and selected

  Default search knobs now are:

  - SEARCH_ROTATION_OPTIONS=0,1
  - SEARCH_MIXED_PRECISION_BUDGETS=0,524288
  - SEARCH_MIXED_PRECISION_MAX_TENSORS=0,2
  - SEARCH_TARGET_TOTAL_BYTES=16777216
  - SEARCH_MAX_FRONTIER_EVALS=6
  - SELF_CALIB_NUM_SEQS=24
  - SELF_CALIB_SEQ_LEN=512
  - GPTQ_BLOCK_SIZE=128
  - ROTATION_AWARE_ENABLED=1
  - ROTATION_BLOCK_SIZE=128
  - HESSIAN_DAMPING=0.01
  - MIXED_PRECISION_EXTRA_BUDGET_BYTES=524288
  - MIXED_PRECISION_MAX_TENSORS=2

  Concretely, this folder keeps the QuantExport2 self-generated calibration search, but upgrades the export search to use those generated sequences for
  more than clip-percentile selection. The search now:

  - builds self-generated sequences
  - collects Hessian proxies from them
  - tries clip-percentile candidates
  - for each candidate, quantizes with GPTQ-style error compensation
  - chooses plain vs rotated quantization per matrix
  - optionally upgrades a few sensitive large tensors to fp16
  - scores the exported model on the calibration sequences and picks the best candidate



In [1]:
!git clone https://github.com/IanniMuliterno/parameter-golf.git

Cloning into 'parameter-golf'...
remote: Enumerating objects: 665, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 665 (delta 10), reused 8 (delta 6), pack-reused 649 (from 3)
Receiving objects: 100% (665/665), 1.43 MiB | 22.54 MiB/s, done.
Resolving deltas: 100% (288/288), done.


In [2]:
%cd parameter-golf/colab/2026-04-06_QuantExport3_RotationAware_GPTQMix

/content/parameter-golf/colab/2026-04-06_QuantExport3_RotationAware_GPTQMix


In [ ]:
 !INT8_CLIP_PERCENTILE=99.99 \
  SELF_CALIB_CANDIDATE_PERCENTILES=99.99,99.999,99.99984 \
  GPTQ_BLOCK_SIZE=64 \
  HESSIAN_DAMPING=0.03 \
  GPTQ_ACCEPT_MSE_RATIO=1.03 \
  ROTATION_AWARE_ENABLED=0 \
  SEARCH_ROTATION_OPTIONS=0 \
  MIXED_PRECISION_EXTRA_BUDGET_BYTES=0 \
  MIXED_PRECISION_MAX_TENSORS=0 \
  SEARCH_MIXED_PRECISION_BUDGETS=0 \
  SEARCH_MIXED_PRECISION_MAX_TENSORS=0 \
  SELF_CALIB_NUM_SEQS=24 \
  SELF_CALIB_SEQ_LEN=512 \
  SELF_CALIB_BATCH_SIZE=4 \
  bash run.sh

manifest.json: 1.93kB [00:00, 7.48MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 124M/124M [00:01<00:00, 68.0MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:01<00:00, 110MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:02<00:00, 99.5MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:02<00:00, 99.4MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:01<00:00, 110MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:01<00:00, 110MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:01<00:00, 124MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:01<00:00, 110MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:01<00:00, 124MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:01<00:00, 110MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:02<00:00, 99.1MB/s]
datasets/tokenizers/fineweb_1024_bpe.mod(…)

In [5]:
!ITERATIONS=300 \
  EXPORT_COMPRESSOR=lzma \
  EXPORT_COMPRESSION_LEVEL=9 \
  INT8_CLIP_PERCENTILE=99.99 \
  INT8_KEEP_FLOAT_MAX_NUMEL=262144 \
  INT8_KEEP_FLOAT_STORE_DTYPE=fp16 \
  INT8_PER_ROW_SCALE_DTYPE=fp16 \
  SELF_CALIB_NUM_SEQS=0 \
  SELF_CALIB_SEQ_LEN=512 \
  SELF_CALIB_BATCH_SIZE=4 \
  SELF_CALIB_CANDIDATE_PERCENTILES=99.99 \
  ROTATION_AWARE_ENABLED=0 \
  SEARCH_ROTATION_OPTIONS=0 \
  MIXED_PRECISION_EXTRA_BUDGET_BYTES=0 \
  MIXED_PRECISION_MAX_TENSORS=0 \
  SEARCH_MIXED_PRECISION_BUDGETS=0 \
  SEARCH_MIXED_PRECISION_MAX_TENSORS=0 \
  GPTQ_BLOCK_SIZE=128 \
  ROTATION_BLOCK_SIZE=128 \
  HESSIAN_DAMPING=0.01 \
  bash run.sh

manifest.json: 1.93kB [00:00, 4.42MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 124M/124M [00:02<00:00, 61.0MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:01<00:00, 110MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:02<00:00, 90.4MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:02<00:00, 99.1MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:02<00:00, 71.1MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:02<00:00, 99.4MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:02<00:00, 76.6MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:02<00:00, 99.3MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:02<00:00, 99.1MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:01<00:00, 110MB/s]
datasets/datasets/fineweb10B_sp1024/fine(…): 100% 200M/200M [00:01<00:00, 110MB/s]
datasets/tokenizers/fineweb_1024_bpe.mo